# 🛹 The Hoverboard
## What *is* a Tool?

**Section 7** of *Search, Extract & Value: The Lightning Fast Assistant* 🚗💨

---

> ### ☁️ This is the **hosted** version.
> Needs an API key, no GPU.
> The other version is **`07_hoverboard_local.ipynb`**.

---

In **Section 5** we watched this model invent a zoning ordinance. Asked about a
parcel it had never seen, it produced a confident, specific, completely
fictional answer — and cited a Planning Commission that does not exist.

Nothing about the model has changed since then. We are about to change one
thing only:

> ### 🎯 We are going to let it go and look.

A **tool** is not a feature of the model. It is not something the vendor built
in. It is a **plain Python function that you wrote**, plus a description good
enough that the model can work out when to call it.

That is the entire idea, and by the end of this notebook you will have built one
from nothing.

```mermaid
flowchart LR
    A["Question"] --> B["Model reads<br/>the tool list"]
    B --> C{"Do I need<br/>a tool?"}
    C -- "yes" --> D["Writes a<br/>tool call"]
    D --> E["YOUR Python<br/>runs it"]
    E --> F["Result goes<br/>back in"]
    F --> B
    C -- "no" --> G["Answers"]
```

---

### What you need

| | |
|---|---|
| ⏱️ | About 10 minutes |
| 🔑 | An API key in your `.env` |
| 🏘️ | The Hill Valley County data *(this notebook builds it if missing)* |

---

# 🚦 LIVE DEMO PATH

## Setup

**This is the only cell where the two versions of this notebook differ.**

In [1]:
import json

from delorean import agent, data, llm, viz

client = llm.get_api_client()
MODEL = llm.api_model()
data.check()          # make sure Hill Valley County exists on this machine


def generate(messages, schemas=None, max_tokens=400):
    """Send messages to the hosted model and return its reply.

    The Anthropic API describes tools slightly differently from the local
    models: a flat {name, description, input_schema} rather than the nested
    {"type": "function", "function": {...}} the open models use. Same
    information, different envelope -- so we translate.
    """
    kwargs = {"model": MODEL, "max_tokens": max_tokens, "messages": messages}
    if schemas:
        kwargs["tools"] = [{
            "name": s["function"]["name"],
            "description": s["function"]["description"],
            "input_schema": s["function"]["parameters"],
        } for s in schemas]
    return client.messages.create(**kwargs)


print(f"Talking to {MODEL}")

Talking to claude-haiku-4-5


---

## Act 1 — 🔧 A tool is just a function

Here it is. Ten lines of ordinary Python that read one row out of the
Appraiser's database.

There is nothing AI about this function. You could have written it in 2005.

In [2]:
def lookup_parcel_naive(parcel_id: str) -> dict | None:
    """Look up one parcel in the Hill Valley County assessment roll."""
    con = data.cama()
    try:
        row = con.execute(
            "SELECT * FROM parcels WHERE parcel_id = ?", (parcel_id,)
        ).fetchone()
    finally:
        con.close()

    if row is None:
        return None

    return {
        "parcel_id": row["parcel_id"],
        "situs_address": row["situs_address"],
        "owner_name": row["owner_name"],
        "neighborhood": row["neighborhood"],
        "year_built": row["year_built"],
        "living_area_sqft": row["living_area_sqft"],
        "lot_size_sqft": row["lot_size_sqft"],
        "assessed_value": row["assessed_value"],
    }


# It works. It is a function.
lookup_parcel_naive("04-127-3891")

{'parcel_id': '04-127-3891',
 'situs_address': '1955 Hilldale Ct',
 'owner_name': 'Biff Tannen',
 'neighborhood': 'Hilldale',
 'year_built': 1971,
 'living_area_sqft': 1640,
 'lot_size_sqft': 15800,
 'assessed_value': 412500}

> ### 💡 That is the whole mystery of "tool calling", solved.
>
> Whatever you can write in Python, the model can use. A database query, a
> spreadsheet, an internal API, a calculation your office has done by hand for
> thirty years. **Your imagination is the only bound.**

---

## Act 2 — 📝 Describe it so a model can choose it

The model cannot see your code. It sees a **description** — and every word of
that description is written for the model to read, not for a developer.

This is a JSON Schema. It says: here is a function, here is what it is for,
here are the arguments it needs.

In [3]:
LOOKUP_PARCEL = {
    "type": "function",
    "function": {
        "name": "lookup_parcel",

        # This sentence is doing the real work. It is how the model decides
        # WHETHER to call this at all, so it is worth writing carefully.
        "description": (
            "Look up one parcel in the Hill Valley County assessment roll by "
            "its parcel number. Returns the owner of record, the situs "
            "address, the property characteristics and the current assessed "
            "value. Use this whenever a question names a specific parcel."
        ),

        "parameters": {
            "type": "object",
            "properties": {
                "parcel_id": {
                    "type": "string",
                    "description": ("The parcel number, formatted NN-NNN-NNNN, "
                                    "for example 04-127-3891."),
                },
            },
            "required": ["parcel_id"],
        },
    },
}

print(json.dumps(LOOKUP_PARCEL, indent=2)[:400] + " ...")

{
  "type": "function",
  "function": {
    "name": "lookup_parcel",
    "description": "Look up one parcel in the Hill Valley County assessment roll by its parcel number. Returns the owner of record, the situs address, the property characteristics and the current assessed value. Use this whenever a question names a specific parcel.",
    "parameters": {
      "type": "object",
      "properties": ...


> 📌 **Write the description like a job posting, not a code comment.** The
> phrase *"use this whenever a question names a specific parcel"* is what stops
> the model reaching for a database to answer "good morning".

---

## Act 3 — 👀 Watch it ask

We hand the model the question **and** the tool description, and see what comes
back.

In [4]:
QUESTION = "Who owns parcel 04-127-3891 and what is it assessed at?"

response = generate([{"role": "user", "content": QUESTION}],
                    schemas=[LOOKUP_PARCEL])

for block in response.content:
    print(f"block type: {block.type}")
    if block.type == "tool_use":
        print(f"  it asked for : {block.name}")
        print(f"  with arguments: {block.input}")
    elif block.type == "text":
        print(f"  it said      : {block.text[:120]}")

print(f"\nstop_reason: {response.stop_reason}")

block type: tool_use
  it asked for : lookup_parcel
  with arguments: {'parcel_id': '04-127-3891'}

stop_reason: tool_use


In [5]:
# Pull the request out into plain variables, so the rest of this notebook is
# identical to the local version.
call = next(b for b in response.content if b.type == "tool_use")
tool_name, tool_args = call.name, dict(call.input)

print("the model asked for :", tool_name)
print("with arguments      :", tool_args)

the model asked for : lookup_parcel
with arguments      : {'parcel_id': '04-127-3891'}


> ### 🔒 One thing this notebook cannot show you
>
> The local version of this notebook shows you the model's **raw output** at
> this point — the literal text it wrote:
>
> ```
> <tool_call>
> <function=lookup_parcel>
> <parameter=parcel_id>
> 04-127-3891
> </parameter>
> </function>
> </tool_call>
> ```
>
> …and then finds it with a two-line regular expression. The point being that
> nothing magic happened: the model wrote text and *our* Python noticed.
>
> **The hosted API does that parsing for you**, and hands back a tidy
> `tool_use` object instead. Genuinely more convenient — and it hides the one
> thing worth seeing, which is that a "tool call" is just text in a format
> everyone agreed on.
>
> If you have a GPU, run `07_hoverboard_local.ipynb` once for that view. 🖥️

---

## Act 4 — 🔁 Run it, and hand the answer back

We now have a function name and some arguments. So we call the function — the
plain Python one from Act 1 — and give the model what it returned.

In [6]:
# 1. Run the actual Python function the model asked for.
result = lookup_parcel_naive(**tool_args)
print("the tool returned:", json.dumps(result, indent=2)[:220], "...\n")

the tool returned: {
  "parcel_id": "04-127-3891",
  "situs_address": "1955 Hilldale Ct",
  "owner_name": "Biff Tannen",
  "neighborhood": "Hilldale",
  "year_built": 1971,
  "living_area_sqft": 1640,
  "lot_size_sqft": 15800,
  "assessed_ ...



In [7]:
# 2. Put the result back into the conversation and ask again.
#    The API wants tool results as a `tool_result` block matching the id of
#    the request. The model has no memory, so it has to be told.
messages = [
    {"role": "user", "content": QUESTION},
    {"role": "assistant", "content": response.content},
    {"role": "user", "content": [{
        "type": "tool_result",
        "tool_use_id": call.id,
        "content": json.dumps(result),
    }]},
]

final = generate(messages, schemas=[LOOKUP_PARCEL])
print(final.content[0].text)

Based on the Hill Valley County assessment roll:

**Owner:** Biff Tannen

**Assessed Value:** $412,500

This property is located at 1955 Hilldale Ct in the Hilldale neighborhood. It's a 1971-built home with 1,640 square feet of living area on a 15,800 square foot lot.


> ### 💡 In Section 5 this exact question got an invented answer.
>
> Same model. Same weights. The only difference is that this time it was
> allowed to look, and somebody wrote ten lines of Python to let it.

---

## Act 5 — 💥 Now watch it break

That worked because the parcel number came through cleanly. Try it the way a
real person types.

In [8]:
for typed in ["04-127-3891", "04127-3891", "4-127-3891"]:
    found = lookup_parcel_naive(typed)
    print(f"lookup_parcel_naive({typed!r:<16}) -> "
          f"{found['owner_name'] if found else 'NOTHING FOUND'}")

lookup_parcel_naive('04-127-3891'   ) -> Biff Tannen
lookup_parcel_naive('04127-3891'    ) -> NOTHING FOUND
lookup_parcel_naive('4-127-3891'    ) -> NOTHING FOUND


> ### ⚠️ Two of those are the same parcel. Our function found one.
>
> And this is not hypothetical. We asked the model the same question ten
> different ways and it emitted the parcel number **three different ways** —
> faithfully echoing whatever the user typed:
>
> | The user typed | The model sent |
> |---|---|
> | `parcel 04-127-3891` | `'04-127-3891'` ✅ |
> | `parcel 04 127 3891` | `'04-127-3891'` ✅ |
> | `04127-3891` | `'04127-3891'` ⚠️ |
> | `4-127-3891` | `'4-127-3891'` ⚠️ |
>
> A plain `WHERE parcel_id = ?` answers **8 of those 10 questions** — and fails
> *silently*. "No such parcel" looks exactly like a parcel that does not exist.

### 🎯 And we already know why.

**Section 2.** We watched `parcel 04-127-3891` shatter into **thirteen separate
tokens** — every digit its own fragment. A model that sees fragments cannot be
trusted to put them back together the same way twice.

This is the bill for that, arriving eighty minutes later.

---

## Act 6 — 🛡️ The same tool, hardened

`src/delorean/tools/` has the grown-up version. Same query underneath — but it
repairs the identifier before it goes anywhere near the database.

In [9]:
from delorean import tools
from delorean.tools.parcel import normalize_parcel_id

for typed in ["04-127-3891", "04127-3891", "4-127-3891",
              "parcel #04 127 3891", "3891"]:
    print(f"{typed!r:<22} -> {normalize_parcel_id(typed)}")

'04-127-3891'          -> 04-127-3891
'04127-3891'           -> 04-127-3891
'4-127-3891'           -> 04-127-3891
'parcel #04 127 3891'  -> 04-127-3891
'3891'                 -> None


Extract the digits, pad to nine, put the hyphens back. The format is
fixed-width, so there is only one way eight digits can become nine.

Note the last one. `"3891"` is a **fragment**, not a parcel number, and it comes
back `None` — the tool refuses rather than guessing. Inventing a plausible
parcel is exactly the behaviour Section 5 spends three minutes warning about.

In [10]:
# The hardened tool answers all three spellings identically.
for typed in ["04-127-3891", "04127-3891", "4-127-3891"]:
    out = json.loads(tools.dispatch("lookup_parcel", {"parcel_id": typed}))
    print(f"{typed!r:<16} -> ok={out['ok']}  {out['data']['owner_name']}")

'04-127-3891'    -> ok=True  Biff Tannen
'04127-3891'     -> ok=True  Biff Tannen
'4-127-3891'     -> ok=True  Biff Tannen


In [11]:
# And when something genuinely IS wrong, the failure is written for the MODEL
# to read -- because that message is what it reasons from next.
print(json.dumps(
    json.loads(tools.dispatch("lookup_parcel", {"parcel_id": "04-127-9999"})),
    indent=2))

{
  "ok": false,
  "error": "No parcel 04-127-9999 on the Hill Valley County roll.",
  "hint": "Parcels that do exist in that book and page: 04-127-3027, 04-127-3891. Check the number and try again. You can also search by owner or address with search_parcels."
}


> ### 💡 Errors are prompts.
>
> A stack trace teaches a model nothing. *"Parcels that do exist in that book
> and page: 04-127-3027, 04-127-3891"* lets it correct itself on the next turn.
>
> This is the single most useful idea in the whole repository, and it is worth
> stealing whatever you build.

---

## Act 7 — ♻️ Put it in a loop

So far we have done every step by hand: ask, parse, run, hand back, ask again.

An **agent** is that loop, automated. Two nodes and one decision:

```mermaid
graph TD;
    __start__([start]):::first
    reason(reason)
    act(act)
    __end__([end]):::last
    __start__ --> reason;
    act --> reason;
    reason -. "no tool needed" .-> __end__;
    reason -.-> act;
    classDef default fill:#f2f0ff,line-height:1.2
    classDef first fill-opacity:0
    classDef last fill:#bfb6fc
```

That is **Reason → Act → Observe** from Section 6, and it is about forty lines
in `src/delorean/agent.py`. Nothing is hidden.

In [12]:
office_agent = agent.build_agent(
    agent.api_responder(client, MODEL),
    tool_names=["lookup_parcel"],
)

# The graph draws its own diagram. No extra library, no network call.
print(office_agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	reason(reason)
	act(act)
	__end__([<p>__end__</p>]):::last
	__start__ --> reason;
	act --> reason;
	reason -. &nbsp;stop&nbsp; .-> __end__;
	reason -.-> act;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [13]:
state = agent.ask(office_agent, "Pull up 4-127-3891.")

viz.show_trace(state["trace"],
               answer=agent.final_answer(state),
               question="Pull up 4-127-3891.")

parcel id,04-127-3891
situs address,1955 Hilldale Ct
owner name,Biff Tannen
neighborhood,Hilldale
property class,Single Family Residential
year built,1971
living area sqft,1640
lot size sqft,15800
bedrooms,3
bathrooms,2.0
assessed value,412500


> ### 🎯 Read that trace top to bottom.
>
> It **reasoned** that it needed a tool, **acted** by asking for one,
> **observed** what came back, and then answered. With a parcel number that was
> missing its leading zero the whole way through.

---

## 🎬 That's Section 7

1. 🔧 **A tool is a Python function you gave it permission to call.** Not a
   vendor feature. Not magic. Yours.
2. 📝 **The description is the interface.** It is how the model decides whether
   to call it, so write it for a reader, not a compiler.
3. 🛡️ **Small models are precise about *which* tool and sloppy about *how*.**
   Absorb the sloppiness in your code and they behave far better than their
   size suggests.
4. 💬 **Errors are prompts.** Every failure message is the model's next
   instruction.

> ### ⏭️ **Next: Section 9 — Reaching 88 MPH.**
> One tool is a parlour trick. Three tools pointed at a real valuation problem
> is an assistant.

---

---

# 🔬 EXPLORE ON YOUR OWN

*Off the live path — for when you take this back to the office.*

---

## 8. 🌦️ The simplest possible tool

Section 5 asked this model for the weather and it correctly said it could not
know. Here is the fix, in six lines. It returns invented data — swap the body
for a real weather API and it is a real tool.

In [14]:
WEATHER = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": ("Get the current weather for a city. Use this for any "
                        "question about current conditions or temperature."),
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string",
                                    "description": "City name, e.g. Calgary"}},
            "required": ["city"],
        },
    },
}


def get_weather(city: str) -> dict:
    """Pretend to be a weather service. Replace this body with a real API."""
    return {"city": city, "temperature_c": 21, "conditions": "Sunny",
            "note": "INVENTED DATA - this is a demonstration tool"}


print(get_weather("Calgary"))

{'city': 'Calgary', 'temperature_c': 21, 'conditions': 'Sunny', 'note': 'INVENTED DATA - this is a demonstration tool'}


The point is not the weather. The point is that **you now know how to add
anything.** A CAMA query, a permit lookup, a sketch-area calculator, an
Esri REST endpoint. Write the function, write the description, hand it over.

---

## 9. 🌐 A tool that reaches the live internet

`tavily-python` is already installed. This one genuinely searches the web —
which is why it is **not** on the live demo path: it needs internet *and* an
API key, and conference wifi is the single biggest risk to a talk.

Set `TAVILY_API_KEY` in your `.env` and this runs.

In [15]:
import os

if os.environ.get("TAVILY_API_KEY"):
    from tavily import TavilyClient

    tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
    hits = tavily.search("IAAO assessment standards", max_results=2)
    for hit in hits["results"]:
        print(f"- {hit['title']}\n  {hit['url']}")
else:
    print("No TAVILY_API_KEY set - skipping.")
    print("Get a free key at https://tavily.com and put it in your .env")

- IAAO Technical Standards | IAAO
  https://www.iaao.org/industry-data/iaao-technical-standards
- Courses | IAAO
  https://www.iaao.org/education/courses


---

## 10. 🎛️ Knowing when *not* to use a tool

An agent that reaches for the database every time somebody says "good morning"
is worse than useless. Watch what the same agent does with a question it should
simply answer.

In [16]:
state = agent.ask(
    office_agent,
    "What is the difference between market value and assessed value?",
)

used = [e["tool"] for e in state["trace"] if e["step"] == "act"]
print("tools used:", used or "none - it just answered")
print()
print(agent.final_answer(state)[:400])

tools used: none - it just answered

# Market Value vs. Assessed Value

These are two important but different concepts in real estate:

## **Market Value**
- The price at which a property would sell under normal market conditions
- Determined by supply and demand, comparable sales, and actual buyer-seller negotiations
- What a willing buyer would pay a willing seller in an arm's-length transaction
- Can fluctuate based on market cond


> 📌 **This gets harder as you add tools.** We measured it: with one or two
> tools this model showed perfect restraint, and at three it started searching
> the policy manual to answer questions it already knew.
>
> The fix is not fewer tools — it is a **system prompt** that says when to
> reach and when to answer. Section 4's lesson, doing real engineering work.
> The Streamlit app for this section lets you switch it on and off and watch
> the difference.

---

<div align="center">

### 🕰️ *"Hey McFly! I thought I told you never to come in here."* 🕰️
### *Bring a hoverboard next time.*

🚗💨 🛹 🏘️

</div>